# SUSENAS Jawa Barat — Analisis Kerawanan Pangan 2019-2023

> **Environment:** Google Colab (R kernel)
> **Repo:** https://github.com/Emanxaa/susenas-datviz
> **Data:** Google Drive BPS — SUSENAS JAWA BARAT 2019-2023


In [ ]:
# ============================================================
# STEP 1: Mount Google Drive
# ============================================================
# Jalankan cell ini PERTAMA. Akan muncul popup untuk authorize.
library(reticulate)
py_run_string("
import sys
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')
")

In [ ]:
# ============================================================
# STEP 2: Clone repo & setup R environment
# ============================================================
system('git clone https://github.com/Emanxaa/susenas-datviz /content/susenas-datviz 2>&1', intern=TRUE)
setwd('/content/susenas-datviz')
cat('Working dir:', getwd(), '\n')

# Install packages yang diperlukan
pkgs <- c('DBI', 'RSQLite', 'tidyverse', 'data.table', 'foreign', 'readxl', 'ggrepel')
new_pkgs <- pkgs[!sapply(pkgs, requireNamespace, quietly=TRUE)]
if (length(new_pkgs) > 0) {
  cat('Installing:', paste(new_pkgs, collapse=', '), '\n')
  install.packages(new_pkgs, quiet=TRUE)
}
cat('Packages OK\n')

In [ ]:
# ============================================================
# STEP 3: Deteksi lokasi data di Google Drive
# ============================================================
# Sesuaikan DRIVE_ROOT dengan path folder SUSENAS di Google Drive Anda
# Contoh: jika folder ada di 'My Drive/SUSENAS/JAWA BARAT/'

DRIVE_ROOT <- '/content/drive/MyDrive/SUSENAS/JAWA BARAT'

# Cek apakah path benar
if (!dir.exists(DRIVE_ROOT)) {
  # Coba cari otomatis
  possible <- system("find /content/drive -type d -name 'JAWA BARAT' 2>/dev/null", intern=TRUE)
  if (length(possible) > 0) {
    DRIVE_ROOT <- possible[1]
    cat('Auto-detected DRIVE_ROOT:', DRIVE_ROOT, '\n')
  } else {
    stop('Folder SUSENAS/JAWA BARAT tidak ditemukan di Google Drive. Cek path DRIVE_ROOT.')
  }
}

cat('DRIVE_ROOT:', DRIVE_ROOT, '\n')

# Tampilkan isi folder
cat('\nFolder yang tersedia:\n')
cat(paste(list.dirs(DRIVE_ROOT, recursive=FALSE), collapse='\n'), '\n')

In [ ]:
# ============================================================
# STEP 4: Ingest data ke SQLite (lazy — skip jika sudah ada)
# ============================================================
source('R/utils.R')
source('R/03_import_sqlite.R')

# Ingest semua tahun yang tersedia di Drive
for (yr in c(2019, 2020, 2021, 2022, 2023)) {
  yr_dir <- file.path(DRIVE_ROOT, yr)
  if (dir.exists(yr_dir)) {
    cat(sprintf('\n[%d] Ditemukan di Drive\n', yr))
    import_susenas_year(year = yr, susenas_root = DRIVE_ROOT, force = FALSE)
  } else {
    cat(sprintf('[%d] Tidak ada di Drive\n', yr))
  }
}

# Cek tabel yang ada
con <- get_susenas_con()
cat('\nTabel di SQLite:', paste(dbListTables(con), collapse=', '), '\n')
dbDisconnect(con)

In [ ]:
# ============================================================
# STEP 5: Jalankan analisis kerawanan pangan
# ============================================================
source('export_rawan_pangan_2023.R')

In [ ]:
# ============================================================
# STEP 6: Jalankan komparasi sebelum-sesudah swasembada
# ============================================================
source('export_komparasi_swasembada.R')

In [ ]:
# ============================================================
# STEP 7: Tampilkan grafik output
# ============================================================
library(reticulate)

output_pngs <- list.files('output', pattern='\\.png$', full.names=TRUE)
for (f in output_pngs) {
  cat('\n---', basename(f), '---\n')
  py_run_string(paste0("
from IPython.display import Image, display
display(Image('", f, "'))
"))
}

In [ ]:
# ============================================================
# STEP 8: Simpan database SQLite ke Google Drive (opsional)
# ============================================================
# Jika ingin simpan susenas.db ke Drive agar tidak perlu ingest ulang:

SAVE_DB_TO_DRIVE <- FALSE  # Ganti TRUE jika mau simpan

if (SAVE_DB_TO_DRIVE) {
  drive_db_path <- file.path(dirname(DRIVE_ROOT), 'susenas.db')
  file.copy('database/susenas.db', drive_db_path, overwrite=TRUE)
  cat('Database disimpan ke:', drive_db_path, '\n')
  cat('Size:', round(file.info(drive_db_path)$size/1e9, 2), 'GB\n')
}